# 08 · Module 8: Derivatives

**Question.** Should NovaFin price with ML, and what hedge and VaR does the options book need?

Black-Scholes already explains these prices to within noise, so ML should not beat it; the value is in proving that and using BS Greeks to size the hedge and VaR.

**Data.** `nova_options.csv`: 10,000 options; Market_Price is BS plus unbiased noise of about 3%. Split: random 80/20, plus a check by Underlying. VaR shocks from `nova_market_data.csv`.

**Professor's anchor.** S22 (BS assumptions, Greeks, implied volatility and the smile, hedge ratios) and the option-pricing-with-ML notebook (S22–S23).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("08")

In [ ]:
options = load("options")
market = load("market", parse_dates=["Date"])
options.head()

> Evaluation standard: a naive baseline and a transparent model before any tree; PR-AUC against the base rate, Brier score and a calibration check for classifiers; the threshold chosen on validation data and reported on test data; SHAP or coefficients for the model that drives the decision; a stability check by period, segment or stock.

## 1. BS benchmark (Task 1)

Recompute BS (`finance.black_scholes`) and compare it with the given column and with Market_Price: RMSE, MAE, % error by call/put, moneyness band and maturity band (`evaluate.error_by_segment`).

## 2. Implied volatility

`finance.implied_vol` from Market_Price, plotted against moneyness. A flat cloud with no smile shows the prices hold no structure BS misses.

## 3. ML pricing (Task 2)

Target Market_Price ÷ Strike; inputs Spot ÷ Strike, maturity, volatility, rate, type. XGBoost and an MLP, plus a hybrid predicting the residual (Market − BS). Test RMSE against BS on the same rows. Never use the given BS column as a feature.

## 4. Noise floor

The residual (Market − BS) has no pattern by moneyness, and its size is proportional to price: about 3.5–4% for options above ₹80 (₹2 standard deviation for the cheapest fifth, ₹19 for the dearest; ₹9.8 pooled, see notebook 00). Compare ML with BS by price band: an ML model scoring better than the noise in a band on test data is fitting noise.

## 5. Greeks

Delta, gamma, vega per option (`finance.greeks`), totalled by underlying (one contract per row).

## 6. Hedge

Delta-neutral hedge in the underlying per stock; a beta hedge with index futures for the Module 6 portfolio (S22).

## 7. VaR

1-day 99% VaR by full revaluation, spot shocks from each underlying's history in nova_market_data plus a volatility shock (`finance.historical_var`). Before and after the delta hedge.

## Handoff to Module 9

VaR before and after hedging, capital set aside for hedging and margin, and the residual vega risk.

In [ ]:
# lines = [handoff.make_line(
#     business_line="...", capacity_cr=..., expected_return_pct=..., expected_loss_pct=...,
#     capital_charge_pct=..., stress_loss_pct=..., ml_edge="validated | weak | none", evidence="...",
#     key_risks=[("risk", "control"), ("risk", "control")],
# )]
# handoff.write_handoff("08", lines)

## Traps to check before handing over

Taking a lower in-sample RMSE as evidence that ML prices better; using the given BS column inside the ML features, which hands the model the answer.